# PT-table pilot: arbitrary cosmology → native components → reconstructed multipoles

This notebook is the first executable pilot for the **functional** cosmological $P(k)$ emulator. It tests one reproducible 8D Latin-hypercube cosmology inside the domain expanded 25% beyond the Abacus c130–c181 extrema, generates the native Velocileptors PT component table, saves it with metadata, and attempts an independent reconstruction of the three multipoles.

**Scope:** one cosmology only. This is an inspection/validation pilot, not the 52-cosmology production generator and not the Abacus-anchor comparison.

**Environment assumptions**
- Run from the project checkout in the NERSC Jupyter environment.
- The project package is importable as `src`; homemade modules live in `/src/`.
- Do **not** modify `sys.path` or reset Python paths.
- The active environment already contains the required `desilike`, `cosmoprimo`, and CLASS dependencies.

The eight emulator parameters are `omega_b`, `omega_cdm`, `sigma8_cb`, `n_s`, `alpha_s`, `N_ur`, `w0_fld`, and `wa_fld`. The derived quantities $h$ and $A_s$ are solved by the existing `src.cosmology` module, not sampled independently. Redshift is $z=0.8$.

In [ ]:
from pathlib import Path
import json
import inspect
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=7, suppress=True)


## 1. Project imports and input catalogue

Set only the catalogue path below. The notebook deliberately does not alter the Python path. The catalogue must contain the Abacus emulator grid rows c130–c181 and the eight independent cosmological parameters.

The c130 cosmology is the project's established fiducial:
- $omega_b=0.02237$
- $omega_{\rm cdm}=0.12$
- $\sigma_{8,cb}=0.714197$
- $n_s=0.9649$
- $\alpha_s=0$
- $N_{\rm ur}=2.0328$
- $w_0=-1$
- $w_a=0$

The code reads these values from the catalogue and checks them against the recorded fiducial to catch an incorrect input file.

In [ ]:
# EDIT THIS PATH to the cosmology catalogue on NERSC.
COSMOLOGY_CSV = Path("/path/to/cosmologies.csv")

if not COSMOLOGY_CSV.is_file():
    raise FileNotFoundError(
        f"Set COSMOLOGY_CSV to the Abacus cosmology catalogue; not found: {COSMOLOGY_CSV}"
    )

from src.cosmology import (
    COSMOLOGY_PARAMETERS,
    read_abacus_cosmologies,
    get_cosmology_dict,
)
from src.theory import (
    TheoryManager,
    Z_FID,
    NBAR_FID,
    ELL_FID,
    K_FID,
)

print("Python:", platform.python_version())
print("Catalogue:", COSMOLOGY_CSV)
print("Theory redshift:", Z_FID)
print("Theory output ells:", ELL_FID)
print("Theory output k bins:", len(K_FID), (float(np.min(K_FID)), float(np.max(K_FID))))
print("Cosmology parameters:", COSMOLOGY_PARAMETERS)

assert np.isclose(Z_FID, 0.8), f"Expected z=0.8, got {Z_FID}"
assert list(COSMOLOGY_PARAMETERS) == [
    "omega_b", "omega_cdm", "sigma8_cb", "n_s",
    "alpha_s", "N_ur", "w0_fld", "wa_fld"
]


## 2. Construct the expanded 8D domain

For each parameter, calculate its minimum and maximum over c130–c181. The expanded interval is centered on the c130 fiducial and extends **1.25 times the largest fiducial-to-extremum distance** in either direction. This guarantees that the original Abacus extrema remain inside the expanded interval, even if the fiducial is not exactly at the midpoint of the extrema.

No hard-coded approximate bounds are used.

In [ ]:
GRID_IDS = list(range(130, 182))
catalogue = read_abacus_cosmologies(COSMOLOGY_CSV, cosmology_ids=GRID_IDS)

if len(catalogue) != 52:
    raise ValueError(f"Expected 52 Abacus cosmologies c130-c181; found {len(catalogue)}")

fid_rows = catalogue.loc[catalogue["cosmo_id"] == 130]
if len(fid_rows) != 1:
    raise ValueError("Could not identify exactly one c130 fiducial row.")

fiducial_row = fid_rows.iloc[0]
fiducial = get_cosmology_dict(fiducial_row)

recorded_fiducial = {
    "omega_b": 0.02237,
    "omega_cdm": 0.12000,
    "sigma8_cb": 0.714197,
    "n_s": 0.9649,
    "alpha_s": 0.0,
    "N_ur": 2.0328,
    "w0_fld": -1.0,
    "wa_fld": 0.0,
}
for name, expected in recorded_fiducial.items():
    if not np.isclose(fiducial[name], expected, rtol=0.0, atol=2e-6):
        raise ValueError(
            f"Catalogue c130 {name}={fiducial[name]} does not match "
            f"the recorded fiducial {expected}. Check the catalogue/version."
        )

bounds_rows = []
expanded_bounds = {}
for name in COSMOLOGY_PARAMETERS:
    values = catalogue[name].to_numpy(dtype=float)
    lo, hi = float(values.min()), float(values.max())
    radius = 1.25 * max(fiducial[name] - lo, hi - fiducial[name])
    expanded_bounds[name] = (fiducial[name] - radius, fiducial[name] + radius)
    bounds_rows.append({
        "parameter": name,
        "Abacus_min": lo,
        "fiducial_c130": fiducial[name],
        "Abacus_max": hi,
        "expanded_min": expanded_bounds[name][0],
        "expanded_max": expanded_bounds[name][1],
    })

bounds_df = pd.DataFrame(bounds_rows).set_index("parameter")
display(bounds_df)


## 3. Draw one reproducible arbitrary cosmology

Use a scrambled 8D Latin hypercube with a fixed seed. The pilot point is independent of the Abacus anchors, but lies inside the expanded rectangular domain. We save its exact parameter values; the production design will later use a larger, independently generated 52-point LHS.

In [ ]:
from scipy.stats import qmc

PILOT_SEED = 20261110
sampler = qmc.LatinHypercube(d=len(COSMOLOGY_PARAMETERS), seed=PILOT_SEED)
unit_point = sampler.random(n=1)[0]

pilot_cosmo = {}
for i, name in enumerate(COSMOLOGY_PARAMETERS):
    lo, hi = expanded_bounds[name]
    pilot_cosmo[name] = float(lo + unit_point[i] * (hi - lo))

pilot_cosmo_df = pd.DataFrame([pilot_cosmo])
display(pilot_cosmo_df)

print("Pilot seed:", PILOT_SEED)
print("Pilot lies inside expanded domain:",
      all(expanded_bounds[k][0] <= pilot_cosmo[k] <= expanded_bounds[k][1]
          for k in COSMOLOGY_PARAMETERS))


## 4. Generate the direct theory spectrum and extract the PT table

The existing `TheoryManager` handles the cosmology conversion ($h$ and $A_s$), constructs the $z=0.8$ LRG-like theory, and caches the initialized object. We use a nontrivial but modest set of bias/EFT parameters; stochastic terms are set to zero for this first algebraic reconstruction check so that the separate stochastic normalization convention does not obscure the PT-table contraction.

The primary object to save is `theory.pt.table`. Its expected layout from the inspected implementation is `(n_ell, n_k_internal, 19)`, with multipoles $0,2,4$. The native $k$ coordinate is discovered from the PT object rather than assumed to equal the 36-bin DESI-style output grid.

In [ ]:
pilot_nuisance = {
    "bias": {"b1": 2.0, "b2": 0.5, "bs": -0.2, "b3": 0.0},
    "eft": {"alpha0": 10.0, "alpha2": -5.0, "alpha4": 2.0, "alpha6": 0.0},
    "stochastic": {"sn0": 0.0, "sn2": 0.0, "sn4": 0.0},
}

manager = TheoryManager(max_cache_size=2, z=0.8)
P_direct = manager.evaluate(pilot_cosmo, **pilot_nuisance)
theory, pipeline = manager.get_theory(pilot_cosmo)

pt = theory.pt
pt_table = np.asarray(pt.table, dtype=float)

print("Theory class:", type(theory))
print("PT class:", type(pt))
print("Direct output shape:", P_direct.shape)
print("PT table shape:", pt_table.shape)
print("PT table finite:", bool(np.isfinite(pt_table).all()))
print("Direct spectrum finite:", bool(np.isfinite(P_direct).all()))

if pt_table.ndim != 3 or pt_table.shape[0] != 3 or pt_table.shape[-1] != 19:
    raise ValueError(
        f"Unexpected PT table shape {pt_table.shape}; expected (3, n_k_internal, 19)."
    )

# Print PT attributes and identify 1D native-k candidates matching the table k axis.
k_candidates = []
for name in dir(pt):
    if name.startswith("_"):
        continue
    try:
        value = getattr(pt, name)
    except Exception:
        continue
    if "k" not in name.lower():
        continue
    try:
        arr = np.asarray(value)
    except Exception:
        continue
    if arr.ndim == 1 and arr.size == pt_table.shape[1] and np.issubdtype(arr.dtype, np.number):
        k_candidates.append((name, arr.astype(float)))

print("\nNative-k candidates:")
for name, arr in k_candidates:
    print(f"  pt.{name}: n={arr.size}, range=({arr.min():.7g}, {arr.max():.7g})")

if not k_candidates:
    raise RuntimeError(
        "Could not identify the native PT k grid. Inspect vars(pt) and the installed "
        "Velocileptors implementation before proceeding."
    )

# Prefer an attribute explicitly named 'k'; otherwise use the first matching candidate.
native_k_name, native_k = next(
    ((name, arr) for name, arr in k_candidates if name.lower() == "k"),
    k_candidates[0],
)
if not np.all(np.diff(native_k) > 0):
    raise ValueError(f"pt.{native_k_name} is not strictly increasing.")

print(f"\nUsing native grid pt.{native_k_name}: {native_k.size} samples")
print("Multipoles:", ELL_FID)


## 5. Inspect the exact installed reconstruction implementation

Import the two helper functions inspected in the previous session and print their signatures/source. Also inspect the theory's growth-rate and tracer-normalization attributes. This keeps the notebook tied to the installed `desilike` implementation rather than silently assuming an API detail.

For this first test, stochastic coefficients are zero. The next pilot iteration can add nonzero stochastic terms after confirming the exact `nd` convention used by the installed theory class.

In [ ]:
from desilike.theories.galaxy_clustering.full_shape import (
    _velocileptors_physical_to_standard,
    _velocileptors_combine_bias_terms_spectrum2_poles,
    get_physical_stochastic_settings,
)

print("physical_to_standard:", inspect.signature(_velocileptors_physical_to_standard))
print("combine_bias_terms:", inspect.signature(_velocileptors_combine_bias_terms_spectrum2_poles))

print("\n--- physical-to-standard source ---")
print(inspect.getsource(_velocileptors_physical_to_standard))
print("\n--- combine source ---")
print(inspect.getsource(_velocileptors_combine_bias_terms_spectrum2_poles))

print("\nPotential growth-rate attributes:")
for obj_name, obj in [("theory", theory), ("template", getattr(theory, "template", None)), ("pt", pt)]:
    if obj is None:
        continue
    for attr in dir(obj):
        if attr.startswith("_") or attr.lower() not in {"f", "growth_rate", "fsat", "sigv", "nbar"}:
            continue
        try:
            value = getattr(obj, attr)
            if np.isscalar(value):
                print(f"  {obj_name}.{attr} = {value}")
        except Exception:
            pass

stochastic_settings = get_physical_stochastic_settings("LRG")
print("\nLRG stochastic settings:", stochastic_settings)
print("Theory nbar:", getattr(theory, "nbar", NBAR_FID))


## 6. Attempt the independent native-grid reconstruction

The reconstruction follows the exact two-step factorization:
1. Convert physical nuisance parameters to the standard 11-element Velocileptors coefficient vector.
2. Contract the native `(3, n_k, 19)` PT table against the 19 monomials.

The growth rate is taken from the initialized theory/template object. The helper's default $A=A_{AP}=1$ is retained for this fiducial-coordinate test. If the installed version exposes the growth rate under a different attribute, the diagnostic cell above identifies it; set `GROWTH_RATE` explicitly rather than guessing.

We compare the reconstructed native-grid spectra, interpolated onto the theory's requested output grid, with the direct pipeline output. This comparison includes the theory's final interpolation step but does not train or fit any coefficients.

In [ ]:
# Try the known template growth-rate attribute first; otherwise require an explicit value.
GROWTH_RATE = None
for candidate_obj in (getattr(theory, "template", None), theory, pt):
    if candidate_obj is None:
        continue
    for attr in ("f", "growth_rate"):
        if hasattr(candidate_obj, attr):
            candidate = getattr(candidate_obj, attr)
            if np.isscalar(candidate):
                GROWTH_RATE = float(candidate)
                print(f"Using growth rate from {type(candidate_obj).__name__}.{attr} = {GROWTH_RATE}")
                break
    if GROWTH_RATE is not None:
        break

if GROWTH_RATE is None:
    raise RuntimeError(
        "Growth rate was not found automatically. Use the attribute/source inspection "
        "above to set GROWTH_RATE from the initialized z=0.8 theory."
    )

nuis = {
    **pilot_nuisance["bias"],
    **pilot_nuisance["eft"],
    **pilot_nuisance["stochastic"],
}
standard_pars = _velocileptors_physical_to_standard(
    b1=nuis["b1"], b2=nuis["b2"], bs=nuis["bs"], b3=nuis["b3"],
    alpha0=nuis["alpha0"], alpha2=nuis["alpha2"],
    alpha4=nuis["alpha4"], alpha6=nuis["alpha6"],
    sn0=nuis["sn0"], sn2=nuis["sn2"], sn4=nuis["sn4"],
    f=GROWTH_RATE,
    fsat=float(stochastic_settings["fsat"]),
    sigv=float(stochastic_settings["sigv"]),
    nbar=float(getattr(theory, "nbar", NBAR_FID)),
    A=1.0, A_AP=1.0, rept=False,
)
standard_pars = np.asarray(standard_pars, dtype=float)
print("Standard coefficient vector shape:", standard_pars.shape)
print("Standard coefficient vector:", standard_pars)

P_native_reconstructed = np.asarray(
    _velocileptors_combine_bias_terms_spectrum2_poles(pt_table, standard_pars),
    dtype=float,
)
print("Native reconstructed shape:", P_native_reconstructed.shape)

if P_native_reconstructed.shape != (3, native_k.size):
    raise ValueError(
        f"Unexpected reconstructed shape {P_native_reconstructed.shape}; "
        f"expected {(3, native_k.size)}."
    )

P_native_on_output_k = np.vstack([
    np.interp(K_FID, native_k, row) for row in P_native_reconstructed
])

residual = P_native_on_output_k - P_direct
relative_rms = np.sqrt(np.mean(residual**2)) / max(np.sqrt(np.mean(P_direct**2)), 1e-30)
max_abs = float(np.max(np.abs(residual)))
max_fractional = float(np.max(
    np.abs(residual) / np.maximum(np.abs(P_direct), 1e-12)
))

print(f"Max absolute residual:       {max_abs:.6e}")
print(f"Relative RMS residual:       {relative_rms:.6e}")
print(f"Max pointwise fractional:    {max_fractional:.6e}")


## 7. Plot the direct and reconstructed spectra

The plot is a visual diagnostic only; the numerical residuals above are the actual validation metrics.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)
for i, (ell, ax) in enumerate(zip(ELL_FID, axes)):
    ax.plot(K_FID, P_direct[i], label="direct theory", lw=2)
    ax.plot(K_FID, P_native_on_output_k[i], "--", label="PT-table reconstruction", lw=1.8)
    ax.set_title(fr"$\ell={ell}$")
    ax.set_xlabel(r"$k\ [h\,\mathrm{Mpc}^{-1}]$")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
axes[0].set_ylabel(r"$P_\ell(k)\ [(h^{-1}\,\mathrm{Mpc})^3]$")
fig.suptitle("Direct theory vs. independent PT-table reconstruction")
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), sharex=True)
for i, (ell, ax) in enumerate(zip(ELL_FID, axes)):
    ax.plot(K_FID, residual[i])
    ax.axhline(0.0, color="black", lw=0.8)
    ax.set_title(fr"Residual, $\ell={ell}$")
    ax.set_xlabel(r"$k\ [h\,\mathrm{Mpc}^{-1}]$")
    ax.grid(alpha=0.25)
axes[0].set_ylabel(r"$P_{\rm reconstructed}-P_{\rm direct}$")
fig.tight_layout()
plt.show()


## 8. Save the pilot PT table and reproducibility metadata

The output is a compressed `.npz` file containing the native PT table, its native $k$ coordinate, the direct theory output grid/spectra, the exact input cosmology and nuisance parameters, the expanded domain, the seed, and reconstruction diagnostics.

The output directory is configurable and is created if needed. Keep this pilot separate from the eventual production training-data directory.

In [ ]:
# EDIT if desired. This directory is created; no existing files are overwritten.
OUTPUT_DIR = Path("./training_data/pt_table_pilot")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / "pilot_lhs_seed20261110.npz"

metadata = {
    "purpose": "single-cosmology native Velocileptors PT-table pilot",
    "cosmology": pilot_cosmo,
    "fiducial_c130": fiducial,
    "expanded_bounds": {k: [float(v[0]), float(v[1])] for k, v in expanded_bounds.items()},
    "pilot_seed": PILOT_SEED,
    "redshift": 0.8,
    "ells": list(map(int, ELL_FID)),
    "native_k_attribute": native_k_name,
    "pt_table_shape": list(pt_table.shape),
    "nuisance": pilot_nuisance,
    "growth_rate": GROWTH_RATE,
    "stochastic_settings": {k: float(v) for k, v in stochastic_settings.items()},
    "nbar": float(getattr(theory, "nbar", NBAR_FID)),
    "reconstruction": {
        "max_abs_residual": max_abs,
        "relative_rms_residual": float(relative_rms),
        "max_pointwise_fractional_residual": max_fractional,
    },
}

np.savez_compressed(
    OUTPUT_FILE,
    pt_table=pt_table,
    k_native=native_k,
    ells=np.asarray(ELL_FID, dtype=int),
    cosmology_json=np.asarray(json.dumps(pilot_cosmo)),
    metadata_json=np.asarray(json.dumps(metadata)),
    k_output=np.asarray(K_FID, dtype=float),
    pk_direct=P_direct,
    pk_reconstructed=P_native_reconstructed,
)
print("Saved:", OUTPUT_FILE.resolve())
print("File size [MB]:", OUTPUT_FILE.stat().st_size / 1e6)

# Reload immediately and verify that no information was lost.
with np.load(OUTPUT_FILE, allow_pickle=False) as saved:
    assert np.array_equal(saved["pt_table"], pt_table)
    assert np.array_equal(saved["k_native"], native_k)
    assert np.array_equal(saved["pk_direct"], P_direct)
    print("Reload check passed.")


## 9. Report back: pilot checklist

Please send back the printed outputs from Sections 4–8, especially:

- PT table shape and the identified native-$k$ attribute/range.
- Growth-rate attribute and LRG stochastic settings.
- The printed source/signatures for the two reconstruction helpers if they differ from the inspected version.
- Maximum absolute, relative RMS, and maximum pointwise fractional reconstruction residuals.
- Whether the `.npz` reload check passed.

**Do not launch the 52-cosmology batch yet.** First we want this one arbitrary cosmology to pass the factorization/reconstruction check and to confirm the saved dataset schema. After that, the production generator can reuse this exact extraction and serialization pathway.